### NumPy Exercises: Supermarket Sales

#### 📋 Dataset Information

**Dataset**: Supermarket Sales  
**Source**: Attached file  

**Columns**:
- `Branch`: Store branch identifier where the transaction occurred ('A', 'B', or 'C').
- `Customer type`: Customer membership status ('Member', 'Normal').
- `Gende`r: Customer's gender ('Male' or 'Female').
- `Product line`: Product category purchased (6 categories: Electronic accessories, Fashion accessories, Food and beverages, Health and beauty, Home and lifestyle, Sports and travel).
- `Quantity`: Number of items purchased in the transaction.
- `Total`: Total amount paid by customer in dollars, including tax.
- `Date`: Transaction date in M/D/YYYY format.
- `Rating`: Customer satisfaction rating on a scale of 0.0 to 10.0 (higher is better).

In [1]:
import numpy as np

In [2]:
## NO NEED CODE HERE
# Load the CSV file
data = np.genfromtxt(
    'Data/supermarket_sales.csv',
    delimiter=',',
    skip_header=1,    # Skip header row
    dtype=str         # Load as strings (no preprocessing)
)

# Print basic info
print("="*60)
print("DATASET LOADED")
print("="*60)
print(f"Shape: {data.shape}")
print(f"  Rows: {data.shape[0]} transactions")
print(f"  Columns: {data.shape[1]}")

# Column names
columns = ['Branch', 'Customer type', 'Gender', 'Product line', 
           'Quantity', 'Total', 'Date', 'Rating']

print(f"\nColumns:")
for i, col in enumerate(columns):
    print(f"  [{i}] {col}")

DATASET LOADED
Shape: (1000, 8)
  Rows: 1000 transactions
  Columns: 8

Columns:
  [0] Branch
  [1] Customer type
  [2] Gender
  [3] Product line
  [4] Quantity
  [5] Total
  [6] Date
  [7] Rating


### Task 1: Customer Segmentation Analysis
**Which customer segment (by type and gender) generates the highest revenue per transaction, and how does their rating behavior differ from other segments?** 

**Instruction:**
- 4 customer segments: Member-Female, Member-Male, Normal-Female, Normal-Male.
- Calculate revenue per transaction for each segment.
- Calculate average rating for each segment.
- How each segment compares to the overall average?
   - Calculate deviations 
- Which segment is the "best"? (Based on revenue, rating and count)
   - Weight: 50% revenue, 30% rating, 20% count (volume) (Need normalize to [0.1])

In [3]:
#TODO: your code here

customer_type = data[:, 1]
gender = data[:, 2]
total = data[:, 5].astype(float)
rating = data[:, 7].astype(float)

# 4 customer segments: Member-Female, Member-Male, Normal-Female, Normal-Male.
segments = {
    'Member-Female': (customer_type=='Member') & (gender=='Female'),
    'Member-Male': (customer_type=='Member') & (gender=='Male'),
    'Normal-Female': (customer_type=='Normal') & (gender=='Female'),
    'Normal-Male': (customer_type=='Normal') & (gender=='Male'),
}

overall_revenue = np.mean(total)
print(f'Overall Average Revenue per Transaction: ', overall_revenue)
overall_rating = np.mean(rating)
print(f'Overall Average Customer Rating: ', overall_rating)
print("-" * 95)

# Calculate revenue per transaction for each segment.
segment_stats = {}

for seg_name, mask in segments.items():
    total_revenue = np.sum(total[mask]) # Calculate revenue per transaction for each segment.
    transaction_count = np.sum(mask) # Count transactions in each segment.
    avg_rating = np.mean(rating[mask]) # Calculate average rating for each segment.
    # Calculate average revenue for each segment.
    revenue_per_txn = total_revenue / transaction_count 
    
    segment_stats[seg_name] = {
        'total_revenue': total_revenue,
        'transaction_count': transaction_count,
        'avg_rating': avg_rating,
        'revenue_per_txn': revenue_per_txn
    }

#  Normalize to [0.1]
def normalize(arr):
    return (arr - np.min(arr)) / (np.max(arr) - np.min(arr))

# Calculate deviations
segment_names = np.array(list(segment_stats.keys()))
revenues = np.array([segment_stats[s]['revenue_per_txn'] for s in segment_names])
ratings  = np.array([segment_stats[s]['avg_rating'] for s in segment_names])
counts   = np.array([segment_stats[s]['transaction_count'] for s in segment_names])

rev_norm   = normalize(revenues)
rate_norm  = normalize(ratings)
count_norm = normalize(counts)

scores = 0.5 * rev_norm + 0.3 * rate_norm + 0.2 * count_norm

print()
print(f"{'Segment':<20} {'Rev/Txn':>10} {'Rating':>10} {'ΔRating':>10} {'Count':>8}")
print('-' * 63)

for i in range(len(segment_names)):
    delta = ratings[i] - overall_rating
    print(f'{segment_names[i]:<20} {revenues[i]:>10.2f} {ratings[i]:>10.2f} {delta:>10.2f} {counts[i]:>8}')

print("-" * 95)
print('\nSegment Scores (Normalized & Weighted):')
    
for i in range(len(segment_names)):
    name = segment_names[i]
    rev_n = rev_norm[i]
    rate_n = rate_norm[i]
    count_n = count_norm[i]
    score = scores[i]
        
    print(f'  {name}:')
    print(f'    Normals (Rev/Rate/Count): [{rev_n:.2f}, {rate_n:.2f}, {count_n:.2f}]')
    print(f'  =>  Final Score: {score:.4f}')

# Find and Print Best Segment
best_idx = np.argmax(scores)
print(f"\nThe 'Best' Customer Segment is: {segment_names[best_idx]}")
print(f'   With the highest weighted score of {scores[best_idx]:.4f}.')

Overall Average Revenue per Transaction:  322.966749
Overall Average Customer Rating:  6.9727
-----------------------------------------------------------------------------------------------

Segment                 Rev/Txn     Rating    ΔRating    Count
---------------------------------------------------------------
Member-Female            337.73       6.94      -0.03      261
Member-Male              316.99       6.94      -0.03      240
Normal-Female            332.23       6.99       0.02      240
Normal-Male              305.05       7.02       0.05      259
-----------------------------------------------------------------------------------------------

Segment Scores (Normalized & Weighted):
  Member-Female:
    Normals (Rev/Rate/Count): [1.00, 0.01, 1.00]
  =>  Final Score: 0.7023
  Member-Male:
    Normals (Rev/Rate/Count): [0.37, 0.00, 0.00]
  =>  Final Score: 0.1826
  Normal-Female:
    Normals (Rev/Rate/Count): [0.83, 0.64, 0.00]
  =>  Final Score: 0.6076
  Normal-Male:
    

### Task 2: Product Performance Optimization
**Which product lines are underperforming (below average sales) in which branches, and what is the revenue opportunity if they reached branch-average performance?**

**Instructions**:
1. **18 combinations**: 3 Branches × 6 Product Lines = 18
2. **Average sales per product-branch combination**
3. **Identify underperformers**: Which are below their branch average?
4. **Calculate gap**: How much below average?
5. **Revenue opportunity**: Potential gain if they reached branch average

In [4]:
#TODO: your code here

branch = data[:, 0]
product_line = data[:, 3]
total = data[:, 5].astype(float)

# How many branches are there?
unique_branches = np.unique(branch)

branch_avg = {}

for b in unique_branches:
    mask = (branch == b)
    branch_average = np.mean(total[mask])
    branch_avg[b] = branch_average
    
# How many product lines are there?
unique_product_lines = np.unique(product_line)
total_revenue_opportunity = 0.0

# Print table header
print(f"\n{'Branch':<8} {'Product Line':<25} {'Combo Avg':>12} {'Branch Avg':>12} {'Gap':>11} {'Count':>9} {'Opportunity':>12}")
print("-" * 95)
    
for b in unique_branches:
    branch_average = branch_avg[b]
    for product in unique_product_lines:
        combo_mask = (branch == b) & (product_line == product)
        combo_totals = total[combo_mask]
        transaction_count = len(combo_totals)
        if transaction_count == 0:
            continue
        combo_avg = np.mean(combo_totals)
        if combo_avg < branch_average:
            gap_per_txn = branch_average - combo_avg
            revenue_opportunity = gap_per_txn * transaction_count
            total_revenue_opportunity += revenue_opportunity
            print(f"{b:<8} {product:<25} {combo_avg:>10.2f} $ {branch_average:>10.2f} $  {gap_per_txn:>8.2f} $  {transaction_count:>8}{revenue_opportunity:>13.2f}")
print("-" * 95)
print(f"{'TOTAL REVENUE OPPORTUNITY:':<84} {total_revenue_opportunity:.2f} $")


Branch   Product Line                 Combo Avg   Branch Avg         Gap     Count  Opportunity
-----------------------------------------------------------------------------------------------
A        Electronic accessories        305.29 $     312.35 $      7.07 $        60       424.13
A        Food and beverages            295.92 $     312.35 $     16.44 $        58       953.43
A        Health and beauty             268.04 $     312.35 $     44.32 $        47      2082.89
B        Electronic accessories        310.03 $     319.87 $      9.85 $        55       541.54
B        Fashion accessories           264.73 $     319.87 $     55.14 $        62      3418.78
B        Food and beverages            304.30 $     319.87 $     15.57 $        50       778.74
C        Fashion accessories           331.69 $     337.10 $      5.41 $        65       351.41
C        Health and beauty             319.53 $     337.10 $     17.57 $        52       913.86
C        Home and lifestyle            

### Task 3: High-Value Customer Identification
**What percentage of total revenue comes from the top 20% of transactions, and what are the common characteristics of these high-value transactions?**

**Instructions**:\
**1. Identify the Top 20% of Transactions by Total amount**

**2. Calculate Revenue Concentration**\
Determine what percentage of total revenue is generated by these top 20% of transactions.

**3. Profile High-Value Transaction Characteristics**\
For the top 20% transactions, analyze their common patterns across multiple dimensions. How many transaction happen: 
 - On each branches
 - On each product lines
 - On each customer types
 - On each gender

**4. Compare High-Value vs Overall Distribution**\
Calculate how the characteristics of high-value transactions differ from the overall dataset by comparing percentage distributions across branches, products, customer types, and gender. \
For example, if Branch A represents 40% of high-value transactions but only 33% of all transactions, this indicates Branch A has a premium customer base. 

**5. Calculate Average Metrics for High-Value Segment**\
Compute the average transaction amount, average quantity purchased, and average rating specifically for the top 20% group and compare these averages to the overall dataset averages. For example:
- Top 20% spend 125% MORE per transaction
- Top 20% buy 55% MORE items per transaction
- Top 20% Ratings ....
  
**6. Identify the "Golden Combination"**\
Find the most common combination of characteristics (Branch + Product + Customer Type + Gender) within the high-value segment

**7. Analyze Contribution by Percentile Groups**\
Beyond just the top 20%, how revenue is distributed across all percentile groups (top 20%, 20-40%, 40-60%, 60-80%, bottom 20%) 

In [5]:
total = data[:, 5].astype(float)

# Identify the Top 20% of Transactions by Total amount
threshold = np.percentile(total, 80)
print(f"Top 20% threshold value: ${threshold:.2f}\n")
top20_mask = total >= threshold
top20_transactions = data[top20_mask]

# Calculate Revenue Concentration
total_revenue = np.sum(total)
top20_revenue = np.sum(total[top20_mask])
revenue_concentration = (top20_revenue / total_revenue) * 100

# Profile High-Value Transaction Characteristics
branch = top20_transactions[:, 0]
unique_branches, counts_branches = np.unique(branch, return_counts=True)
print("Top 20% Transactions by Branch:")
for b, c in zip(unique_branches, counts_branches):
    print(f"  {b}: {c}")

product_lines = top20_transactions[:, 3]
unique_pl, counts_pl = np.unique(product_lines, return_counts=True)
print("\nTop 20% Transactions by Product Line:")
for p, c in zip(unique_pl, counts_pl):
    print(f"  {p}: {c}")

customer_type = top20_transactions[:, 1]
unique_ct, counts_ct = np.unique(customer_type, return_counts=True)
print("\nTop 20% Transactions by Customer Type:")
for ct, c in zip(unique_ct, counts_ct):
    print(f"  {ct}: {c}")

gender = top20_transactions[:, 2]
unique_g, counts_g = np.unique(gender, return_counts=True)
print("\nTop 20% Transactions by Gender:")
for g, c in zip(unique_g, counts_g):
    print(f"  {g}: {c}")

# Compare High-Value vs Overall Distribution
def compare_distribution(overall_col, top_col, label):
    # Overall distribution
    uniq_all, cnt_all = np.unique(overall_col, return_counts=True)
    pct_all = cnt_all / len(overall_col) * 100
    # High-value distribution
    uniq_top, cnt_top = np.unique(top_col, return_counts=True)
    pct_top = cnt_top / len(top_col) * 100
    print(f"\n==== {label} DISTRIBUTION ====")
    for item in uniq_all:
        overall_percent = pct_all[uniq_all == item][0]
        # If category appears in high-value
        if item in uniq_top:
            top_percent = pct_top[uniq_top == item][0]
        else:
            top_percent = 0
        diff = top_percent - overall_percent
        print(f"{item}: Overall = {overall_percent:.2f}% | High-Value = {top_percent:.2f}% | Δ = {diff:+.2f}%")

compare_distribution(
    overall_col = data[:, 0],
    top_col     = top20_transactions[:, 0],
    label       = "BRANCH"
)
compare_distribution(
    overall_col = data[:, 3],
    top_col     = top20_transactions[:, 3],
    label       = "PRODUCT"
)
compare_distribution(
    overall_col = data[:, 1],
    top_col     = top20_transactions[:, 1],
    label       = "CUSTOMER TYPE"
)
compare_distribution(
    overall_col = data[:, 2],
    top_col     = top20_transactions[:, 2],
    label       = "GENDER"
)

# Calculate Average Metrics for High-Value Segment
quantity = data[:, 4].astype(float)
rating = data[:, 7].astype(float)
overall_avg_total = np.mean(total)
overall_avg_quantity = np.mean(quantity)
overall_avg_rating = np.mean(rating)
#--------------
top_avg_total = np.mean(total[top20_mask])
top_avg_quantity = np.mean(quantity[top20_mask])
top_avg_rating = np.mean(rating[top20_mask])
#--------------
pct_more_total = (top_avg_total / overall_avg_total - 1) * 100
pct_more_quantity = (top_avg_quantity / overall_avg_quantity - 1) * 100
pct_more_rating = (top_avg_rating / overall_avg_rating - 1) * 100
#--------------
print("\n=== AVERAGE METRICS COMPARISON ===")
print(f"→ Top 20% spend {pct_more_total:.1f}% MORE per transaction")
print(f"→ Top 20% buy {pct_more_quantity:.1f}% MORE items per transaction")
print(f"→ Top 20% rate {pct_more_rating:.1f}% {'HIGHER' if pct_more_rating>0 else 'LOWER'} than average")

# Identify the "Golden Combination"
top_combined = np.array([
    f"{b}|{p}|{t}|{g}"
    for b, p, t, g in zip(branch, product_lines, customer_type, gender)
])
unique_combos, counts = np.unique(top_combined, return_counts=True)
max_index = np.argmax(counts)
golden_combo = unique_combos[max_index]
golden_count = counts[max_index]
print("\n=== GOLDEN COMBINATION (Top 20% customers) ===")
print(f"Most common combination:")
print(f"  {golden_combo.replace('|', ' - ')}")
print(f"Appears {golden_count} times in top-value transactions")

# Analyze Contribution by Percentile Groups
p20 = np.percentile(total, 20)
p40 = np.percentile(total, 40)
p60 = np.percentile(total, 60)
p80 = np.percentile(total, 80)
bottom20_mask = total <= p20
_20_40_mask    = (total > p20) & (total <= p40)
_40_60_mask    = (total > p40) & (total <= p60)
_60_80_mask    = (total > p60) & (total <= p80)
masks = {
    "Bottom 20%" : bottom20_mask,
    "20–40%"     : _20_40_mask,
    "40–60%"     : _40_60_mask,
    "60–80%"     : _60_80_mask,
    "Top 20%"    : top20_mask
}
total_revenue = total.sum()
print("\n=== Revenue Contribution by Percentile Groups ===\n")
for name, mask in masks.items():
    group_revenue = total[mask].sum()
    group_count   = mask.sum()
    contribution  = group_revenue / total_revenue * 100

    print(f"{name}:")
    print(f"  Transactions: {group_count}")
    print(f"  Revenue:      ${group_revenue:,.2f}")
    print(f"  Contribution: {contribution:.2f}%\n")

Top 20% threshold value: $533.26

Top 20% Transactions by Branch:
  A: 60
  B: 67
  C: 73

Top 20% Transactions by Product Line:
  Electronic accessories: 37
  Fashion accessories: 31
  Food and beverages: 31
  Health and beauty: 32
  Home and lifestyle: 33
  Sports and travel: 36

Top 20% Transactions by Customer Type:
  Member: 104
  Normal: 96

Top 20% Transactions by Gender:
  Female: 104
  Male: 96

==== BRANCH DISTRIBUTION ====
A: Overall = 34.00% | High-Value = 30.00% | Δ = -4.00%
B: Overall = 33.20% | High-Value = 33.50% | Δ = +0.30%
C: Overall = 32.80% | High-Value = 36.50% | Δ = +3.70%

==== PRODUCT DISTRIBUTION ====
Electronic accessories: Overall = 17.00% | High-Value = 18.50% | Δ = +1.50%
Fashion accessories: Overall = 17.80% | High-Value = 15.50% | Δ = -2.30%
Food and beverages: Overall = 17.40% | High-Value = 15.50% | Δ = -1.90%
Health and beauty: Overall = 15.20% | High-Value = 16.00% | Δ = +0.80%
Home and lifestyle: Overall = 16.00% | High-Value = 16.50% | Δ = +0.50%
S